# Objetivo: compor um prato com proteína, carboidrato e vegetal com a menor quantidade de calorias

In [7]:
from dataclasses import dataclass
from enum import Enum
from itertools import cycle
import random
import itertools
from pprint import pprint

# ======== PARAMETROS ========
dias_cardapio = 5
orcamento_maximo_prato = 40
meta_calorias = 700
torelancia_caloria = 50
quantidade_ingreditens = 25

# ======== FUNCOES ========
class TipoIngrediente(Enum):
    PROTEINA = "proteina"
    CARBOIDRATO = "carboidrato"
    VEGETAL = "vegetal"


@dataclass
class Ingrediente:
    nome: str
    preco: float
    caloria: int
    tipo: TipoIngrediente


def gerar_mocks_ingredientes(quantidade: int) -> list[Ingrediente]:
    tipos = list(TipoIngrediente)
    tipos_distribuidos = cycle(tipos)

    ingredientes = []

    for i in range(quantidade):
        tipo = next(tipos_distribuidos)

        ingrediente = Ingrediente(
            nome=f"Ingrediente {i + 1}",
            preco=round(random.uniform(2.0, 30.0), 2),
            caloria=random.randint(20, 500),
            tipo=tipo
        )

        ingredientes.append(ingrediente)

    return ingredientes


def otimizar_lista_otima(ingredientes: list[Ingrediente]):
    combinacoes = itertools.product([0, 1], repeat=quantidade_ingreditens)

    combinacoes_possiveis = []

    for combinacao in combinacoes:
        if combinacao.count(1) != 3:
            continue

        calorias = 0
        preco = 0
        tipos_utilizados = []
        ingredientes_selecionados = []
        for i in range(quantidade_ingreditens):
            if combinacao[i] == 1:
                calorias += ingredientes[i].caloria
                preco += ingredientes[i].preco
                tipos_utilizados.append(ingredientes[i].tipo)
                ingredientes_selecionados.append(ingredientes[i])

        if not TipoIngrediente.VEGETAL in tipos_utilizados:
            continue

        if not TipoIngrediente.CARBOIDRATO in tipos_utilizados:
            continue

        if not TipoIngrediente.PROTEINA in tipos_utilizados:
            continue

        if not meta_calorias - torelancia_caloria <= calorias <= meta_calorias + torelancia_caloria:
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": ingredientes_selecionados,
            "calorias": calorias,
            "preco": preco
        }

        combinacoes_possiveis.append(combinacao_info)

    if len(combinacoes_possiveis) < dias_cardapio:
        return None

    combinacoes_possiveis.sort(key=lambda combinacao: combinacao["preco"])

    return combinacoes_possiveis[0: dias_cardapio]






def otimizar_lista_heuristico(ingredientes: list[Ingrediente]):
    caloria_ideal_carbo = meta_calorias * 0.5
    caloria_ideal_proteina = meta_calorias * 0.4
    caloria_ideal_vegetal = meta_calorias * 0.1

    ingredientes_candidatos = {"carbo": [], "proteina": [], "vegetal": []}

    for ingrediente in ingredientes:
        if ingrediente.tipo == TipoIngrediente.VEGETAL and caloria_ideal_vegetal - torelancia_caloria <= ingrediente.caloria <= caloria_ideal_vegetal + torelancia_caloria:
            ingredientes_candidatos["vegetal"].append(ingrediente)

        if ingrediente.tipo == TipoIngrediente.CARBOIDRATO and caloria_ideal_carbo - torelancia_caloria <= ingrediente.caloria <= caloria_ideal_carbo + torelancia_caloria:
            ingredientes_candidatos["carbo"].append(ingrediente)

        if ingrediente.tipo == TipoIngrediente.PROTEINA and caloria_ideal_proteina - torelancia_caloria <= ingrediente.caloria <= caloria_ideal_proteina + torelancia_caloria:
            ingredientes_candidatos["proteina"].append(ingrediente)



    pratos_candidatos = []
    limite_maximo = 1000
    tentativas = 0
    while tentativas <= limite_maximo:
        tentativas +=1

        carbo_aleatorio = random.choice(ingredientes_candidatos["carbo"])
        proteina_aleatoria = random.choice(ingredientes_candidatos["proteina"])
        vegetal_aleatorio = random.choice(ingredientes_candidatos["vegetal"])

        calorias = carbo_aleatorio.caloria + proteina_aleatoria.caloria + vegetal_aleatorio.caloria
        preco = carbo_aleatorio.preco + proteina_aleatoria.preco + vegetal_aleatorio.preco

        if not meta_calorias - torelancia_caloria <= calorias <= meta_calorias + torelancia_caloria:
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": [carbo_aleatorio, proteina_aleatoria, vegetal_aleatorio],
            "calorias": calorias,
            "preco": preco
        }

        pratos_candidatos.append(combinacao_info)

    if len(pratos_candidatos) < dias_cardapio:
        return None

    # .sort para priorizar ingredientes mais baratos
    pratos_candidatos.sort(key=lambda combinacao: combinacao["preco"])

    return pratos_candidatos[0:dias_cardapio]

        




        


# ======== EXECUÇÃO ========
ingredientes = gerar_mocks_ingredientes(quantidade_ingreditens)
cardapio_exaustivo = otimizar_lista_otima(ingredientes)
cardapio_heuristico = otimizar_lista_heuristico(ingredientes)
pprint(cardapio_exaustivo)
pprint(cardapio_heuristico)


[{'calorias': 670,
  'itens': [Ingrediente(nome='Ingrediente 15',
                        preco=5.56,
                        caloria=136,
                        tipo=<TipoIngrediente.VEGETAL: 'vegetal'>),
            Ingrediente(nome='Ingrediente 17',
                        preco=3.46,
                        caloria=373,
                        tipo=<TipoIngrediente.CARBOIDRATO: 'carboidrato'>),
            Ingrediente(nome='Ingrediente 19',
                        preco=2.01,
                        caloria=161,
                        tipo=<TipoIngrediente.PROTEINA: 'proteina'>)],
  'preco': 11.03},
 {'calorias': 692,
  'itens': [Ingrediente(nome='Ingrediente 9',
                        preco=7.13,
                        caloria=158,
                        tipo=<TipoIngrediente.VEGETAL: 'vegetal'>),
            Ingrediente(nome='Ingrediente 17',
                        preco=3.46,
                        caloria=373,
                        tipo=<TipoIngrediente.CARBOIDRATO: 'c